Purpose: Load validated retail source files into PostgreSQL after preprocessing. The notebook performs a few ingestion-specific checks, then bulk loads users, menu items, cleaned transactions, and transaction items using PostgreSQL COPY.

In [1]:
import psycopg2
from pathlib import Path
import glob
import os
import pandas as pd

In [ ]:
# Database credentials are read from environment variables.
DB_NAME = os.getenv("POSTGRES_DB", "retail_store")
DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")

if DB_PASSWORD is None:
    raise RuntimeError(
        "POSTGRES_PASSWORD is not set. "
        "Set it in your local environment before running the notebook."
    )

In [ ]:
def get_csv_files(directory):
    # Return valid CSV files, excluding macOS metadata and empty files

    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory}")

    return [
        file
        for file in sorted(directory.glob("*.csv"))
        if not file.name.startswith("._")
        and file.stat().st_size > 0
    ]

In [ ]:
# count null values in users directory
user_data_dir = Path("../data/raw/users")

for file in get_csv_files(user_data_dir):
    df = pd.read_csv(file)

    print(f"{file.name}: {df.isna().sum().sum()} null values")

users_202307.csv null values: 0
users_202308.csv null values: 0
users_202309.csv null values: 0
users_202310.csv null values: 0
users_202311.csv null values: 0
users_202312.csv null values: 0
users_202401.csv null values: 0
users_202402.csv null values: 0
users_202403.csv null values: 0
users_202404.csv null values: 0
users_202405.csv null values: 0
users_202406.csv null values: 0
users_202407.csv null values: 0
users_202408.csv null values: 0
users_202409.csv null values: 0
users_202410.csv null values: 0
users_202411.csv null values: 0
users_202412.csv null values: 0
users_202501.csv null values: 0
users_202502.csv null values: 0
users_202503.csv null values: 0
users_202504.csv null values: 0
users_202505.csv null values: 0
users_202506.csv null values: 0


User and menu-item files are inspected for missing values before loading. In the PostgreSQL COPY statements below, NULL '' tells PostgreSQL to interpret empty CSV fields as SQL NULL, including nullable date and numeric columns.

In [ ]:
menu_items_file = Path("../data/raw/menu_items/menu_items.csv")

menu_items_df = pd.read_csv(menu_items_file)

menu_items_df[
    ["available_from", "available_to"]
].isna().sum()

Available_to and Available_from contain all empty strings, read as null values in Pandas. Postgres does not read empty strings as null since the fields are numeric data types, so must convert empty strings. In COPY, will use NULL '' command. 

In [ ]:
transaction_sample = Path(
    "../data/processed/transactions_clean/transactions_202307.csv"
)

transaction_sample_df = pd.read_csv(transaction_sample)

transaction_sample_df.info()

In [ ]:
transaction_items_sample = Path(
    "../data/raw/transaction_items/transaction_items_202307.csv"
)

transaction_items_df = pd.read_csv(transaction_items_sample)

num_duplicates = transaction_items_df.duplicated(
    subset=["transaction_id", "item_id"],
    keep=False,
).sum()

print(
    "Duplicate transaction-item combinations: "
    f"{num_duplicates:,}"
)

Duplicate (transaction_id, item_id) combinations showed that these two columns could not reliably identify individual line-item records. The final schema therefore uses an identity-generated transaction_item_id as the primary key.

## Load Data into PostgreSQL

The validated source files are bulk-loaded into their corresponding PostgreSQL tables using `COPY FROM STDIN`.

In [ ]:
conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
)

cur = conn.cursor()

print("Connected to PostgreSQL.")

## Users

In [ ]:
user_data_dir = Path("../data/raw/users")

for file in get_csv_files(user_data_dir):
    print(f"Loading {file.name} into users")

    with file.open("r", encoding="utf-8") as csv_file:
        cur.copy_expert(
            """
            COPY users
            FROM STDIN
            WITH (
                FORMAT CSV,
                HEADER TRUE,
                NULL ''
            )
            """,
            csv_file,
        )

    conn.commit()

print("Users loaded successfully.")

Loading users_202307.csv
Loading users_202308.csv
Loading users_202309.csv
Loading users_202310.csv
Loading users_202311.csv
Loading users_202312.csv
Loading users_202401.csv
Loading users_202402.csv
Loading users_202403.csv
Loading users_202404.csv
Loading users_202405.csv
Loading users_202406.csv
Loading users_202407.csv
Loading users_202408.csv
Loading users_202409.csv
Loading users_202410.csv
Loading users_202411.csv
Loading users_202412.csv
Loading users_202501.csv
Loading users_202502.csv
Loading users_202503.csv
Loading users_202504.csv
Loading users_202505.csv
Loading users_202506.csv


## Menu Items

In [ ]:
menu_items_data_dir = Path("../data/raw/menu_items")

for file in get_csv_files(menu_items_data_dir):
    print(f"Loading {file.name} into menu_items")

    with file.open("r", encoding="utf-8") as csv_file:
        cur.copy_expert(
            """
            COPY menu_items
            FROM STDIN
            WITH (
                FORMAT CSV,
                HEADER TRUE,
                NULL ''
            )
            """,
            csv_file,
        )

    conn.commit()

print("Menu items loaded successfully.")

## Transactions

In [ ]:
transactions_data_dir = Path(
    "../data/processed/transactions_clean"
)

for file in get_csv_files(transactions_data_dir):
    print(f"Loading {file.name} into transactions")

    with file.open("r", encoding="utf-8") as csv_file:
        cur.copy_expert(
            """
            COPY transactions
            FROM STDIN
            WITH (
                FORMAT CSV,
                HEADER TRUE,
                NULL ''
            )
            """,
            csv_file,
        )

    conn.commit()

print("Transactions loaded successfully.")

## Transaction Items

In [ ]:
transaction_items_data_dir = Path(
    "../data/raw/transaction_items"
)

for file in get_csv_files(transaction_items_data_dir):
    print(f"Loading {file.name} into transaction_items")

    with file.open("r", encoding="utf-8") as csv_file:
        cur.copy_expert(
            """
            COPY transaction_items (
                transaction_id,
                item_id,
                quantity,
                unit_price,
                subtotal,
                created_at
            )
            FROM STDIN
            WITH (
                FORMAT CSV,
                HEADER TRUE,
                NULL ''
            )
            """,
            csv_file,
        )

    conn.commit()

print("Transaction items loaded successfully.")

In [ ]:
cur.close()
conn.close()

print("PostgreSQL connection closed.")